# Sentiment Agent

The **Sentiment Agent** runs after the News Agent (it needs `news_summary` as input).

Uses **gpt-4o-mini** to convert the 3-bullet news summary into a single numeric signal:

| Score | Label |
|---|---|
| 0.0 – 0.2 | Very Bearish |
| 0.2 – 0.4 | Bearish |
| 0.4 – 0.6 | Neutral |
| 0.6 – 0.8 | Bullish |
| 0.8 – 1.0 | Very Bullish |

**Output state key:** `sentiment_score` (float 0.0–1.0)

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv()

_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def _score_to_label(score: float) -> str:
    if score < 0.2:
        return "Very Bearish"
    elif score < 0.4:
        return "Bearish"
    elif score < 0.6:
        return "Neutral"
    elif score < 0.8:
        return "Bullish"
    return "Very Bullish"


def sentiment_node(state: dict) -> dict:
    """
    Score news sentiment with gpt-4o-mini.
    Input: state['news_summary'] (3-bullet string from news_node)
    Returns: sentiment_score (float 0.0–1.0)
    """
    ticker = state["ticker"]
    news_summary = state.get("news_summary") or "No recent news available."
    print(f"[Sentiment] Scoring sentiment for {ticker}...")

    prompt = (
        f"You are a quantitative analyst. Analyze the sentiment of these news bullets "
        f"about {ticker} stock and return a single decimal number between 0.0 and 1.0.\n\n"
        f"Scale: 0.0=Very Bearish, 0.25=Bearish, 0.5=Neutral, 0.75=Bullish, 1.0=Very Bullish\n\n"
        f"News:\n{news_summary}\n\n"
        f"Return ONLY the decimal number — nothing else."
    )

    response = _llm.invoke([HumanMessage(content=prompt)])

    try:
        score = float(response.content.strip())
        score = max(0.0, min(1.0, score))   # clamp to [0, 1]
    except (ValueError, AttributeError):
        score = 0.5  # fallback to neutral

    label = _score_to_label(score)
    print(f"[Sentiment] Score: {score:.2f} ({label})")
    return {"sentiment_score": score}


In [ ]:
if __name__ == "__main__":
    # --- Demo: score a sample news summary ---
    sample_state = {
        "ticker": "AAPL",
        "news_summary": (
            "• Apple reported record Q1 revenue of $124B, beating estimates by 3%\n"
            "• iPhone 16 demand stronger than expected in China despite macro headwinds\n"
            "• Apple Intelligence features driving upgrade cycle acceleration in 2025"
        ),
    }
    result = sentiment_node(sample_state)
    print(f"Sentiment score: {result['sentiment_score']:.2f}")
